# Nirikshak walkthrough

Catching an eavesdropper that the 11% rule never sees, on real IBM Quantum hardware.

This notebook runs on the committed data, so it needs no IBM account:

1. load one hardware batch from `ibm_kingston`,
2. score every block with Nirikshak and with the 11% rule,
3. give every session a verdict,
4. simulate a session pair yourself on Qiskit Aer,
5. run the network monitor on a simulated day.

**Linux:** qiskit-aer and scikit-learn each bundle `libgomp`. Start Jupyter with
`GLIBC_TUNABLES=glibc.rtld.optional_static_tls=4096 jupyter lab` (see the README). This notebook does
not import scikit-learn, so it usually runs without it.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nirikshak.detectors import QBERThreshold, ReferenceConformal, SessionTest
from nirikshak.protocol import bb84_key_rate

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
HW = ROOT / "data" / "hardware"
sorted(p.name for p in HW.glob("*.parquet"))

## 1. One hardware batch

Batch 5 ran on `ibm_kingston` on 22 Oct 2026. It holds 10 sessions: 3 honest, 4 intercept-resend
attacks on 1, 2, 4 or 8 of the 16 rounds per shot, and 3 entangling-probe attacks. Every session has
20 **key** blocks (through the channel, where Eve may act) and 20 **reference** blocks (an honest
calibration circuit on the same qubits, in the same job, that Eve cannot touch). One row per block.

In [ ]:
batch = pd.read_parquet(HW / "dbd0vo424mrc73cki1l0.parquet")
batch[["session_id", "kind", "block", "qber", "n_sifted", "attack", "attacked_fraction"]].head()

## 2. Block alarms: Nirikshak vs the 11% rule

Nirikshak sets each session's threshold from that session's reference blocks (split-conformal,
5% false-alarm budget). The 11% rule is BB84's abort threshold.

In [ ]:
def label(attack, frac, strength):
    if attack == "none":
        return "honest"
    rounds = round(frac * 16)
    return f"intercept {rounds}/16" if attack == "intercept_resend" else f"probe {'pi/8' if strength < 0.5 else 'pi/4'}, {rounds}/16"

key = ReferenceConformal(alpha=0.05).predict_table(batch)
key["rule_11"] = QBERThreshold().predict(key["qber"])
key["condition"] = [label(a, f, s) for a, f, s in zip(key.attack, key.attacked_fraction, key.strength)]
key.groupby("condition", sort=False).agg(key_qber=("qber", "mean"), nirikshak=("alarm", "mean"), rule_11=("rule_11", "mean")).round(3)

## 3. Session verdicts

The decision per session: a one-sided permutation test of key blocks against the session's own
reference blocks. The key rate column is the asymptotic BB84 rate `1 - 2h(QBER)`: while it is above
zero, BB84 keeps making keys and never reports Eve.

In [ ]:
sessions = SessionTest(alpha=0.05).predict_table(batch)
sessions["condition"] = [label(a, f, s) for a, f, s in zip(sessions.attack, sessions.attacked_fraction, sessions.strength)]
sessions["key_rate"] = bb84_key_rate(sessions["key_mean"].to_numpy())
sessions[["condition", "key_mean", "ref_mean", "key_rate", "p_value", "alarm"]].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for i, (sid, g) in enumerate(key.groupby("session_id", sort=False)):
    x = i + np.linspace(-0.3, 0.3, len(g))
    color = "#2a78d6" if g.is_attack.any() else "#898781"
    ax.scatter(x, g.qber, s=18, facecolors=np.where(g.alarm, color, "white"), edgecolors=color)
    ax.hlines(g.threshold.iloc[0], i - 0.4, i + 0.4, color="#eb6834", lw=2)
ax.axhline(0.11, color="#898781", ls="--", lw=1)
ax.text(len(sessions) - 0.5, 0.113, "11% rule", ha="right", color="#555")
ax.set_xticks(range(len(sessions)), sessions.condition, rotation=30, ha="right")
ax.set_ylabel("block QBER")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
ax.set_title("ibm_kingston batch 5: key blocks, session thresholds (orange) and the 11% rule")
plt.tight_layout()

## 4. Simulate a session pair yourself

The same circuits on Qiskit Aer with a simple readout-noise model. Change `readout` or `rounds`
and re-run. Takes about a minute.

In [ ]:
from nirikshak.noise import simple_noise
from nirikshak.session import SessionSpec, simulate_session
from nirikshak.sim import make_simulator

readout, rounds = 0.015, 2   # readout error, rounds per shot Eve intercepts (of 16)
noise = simple_noise(readout=readout, readout_1to0=2 * readout)
sim = pd.concat([
    simulate_session(SessionSpec(attack="none", seed=1), backend=make_simulator(noise), session_id="honest"),
    simulate_session(SessionSpec(attack="intercept_resend", fraction=rounds / 16, seed=2),
                     backend=make_simulator(noise), session_id=f"Eve on {rounds}/16"),
], ignore_index=True)
SessionTest(alpha=0.05).predict_table(sim)[["session_id", "key_mean", "ref_mean", "p_value", "alarm"]].round(4)

## 5. Nirikshak as a network monitor

Per link, the monitor reads telemetry (sifted rounds and errors per block, for key and reference
rounds) and gives an hourly verdict. Links L1-L4 are simulated; L5 replays the real `ibm_kingston`
sessions, one per hour.

In [ ]:
from nirikshak.monitor import NetworkMonitor
from nirikshak.monitor.synthetic import demo_network

records, links = demo_network(HW)
monitor = NetworkMonitor(alpha=0.05).run(records)
links.merge(monitor.status(), on="link_id")[["link_id", "description", "windows", "alarms", "status"]]

## Where next

- `python -m src.main` reruns the stress test, scorecard and figures.
- `results/scorecard.md` has every hardware session with 95% intervals.
- `cd web && npm install && npm run dev` opens the web app.